<a href="https://colab.research.google.com/github/Shubhamspc90/fastapi-by-CampusX/blob/api/fastapi-by-CampusX/insuranceModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [9]:
df = pd.read_csv('/content/sample_data/insurance.csv')

In [10]:
df.sample(5)


,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
43,21,102.8,1.76,40.26,False,Jalandhar,government_job,Medium
69,61,120.4,1.90,24.00,True,Delhi,private_job,High
83,62,61.1,1.91,4.13,True,Lucknow,student,High
55,64,122.9,1.84,26.97,False,Rajkot,business_owner,High
51,64,55.2,1.62,31.96,True,Nagpur,private_job,High


In [11]:
df['occupation'].unique()


array(['government_job', 'retired', 'business_owner', 'private_job',
       'unemployed', 'student', 'freelancer'], dtype=object)

In [12]:
df_feat = df.copy()


In [13]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [14]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [15]:
df_feat["age_group"] = df_feat["age"].apply(age_group)


In [16]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [17]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)


In [19]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [20]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [21]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)


In [22]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)


,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
96,43.53,unemployed,41.954306,young,medium,2,High
27,30.20,retired,52.385028,senior,medium,2,High
78,30.34,private_job,44.414062,adult,high,1,Medium
75,30.78,retired,26.448980,senior,low,3,Medium
79,24.55,government_job,20.003395,adult,low,2,Low


In [23]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [24]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,20.000811,middle_aged,medium,2,39.04,government_job
1,39.901388,middle_aged,high,2,41.73,retired
2,19.317108,adult,low,2,43.90,business_owner
3,30.492182,senior,high,2,24.94,private_job
4,18.950617,adult,low,2,24.47,business_owner
...,...,...,...,...,...,...
95,38.751715,adult,medium,1,46.17,freelancer
96,41.954306,young,medium,2,43.53,unemployed
97,33.644444,adult,high,2,10.04,business_owner
98,23.318800,adult,low,3,4.07,freelancer


In [25]:
y

,insurance_premium_category
0,Medium
1,High
2,Low
3,High
4,Low
...,...
95,Medium
96,High
97,Medium
98,Low


In [26]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [27]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [28]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [29]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [30]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.85

In [31]:
X_test.sample(5)


,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
52,32.594301,young,medium,2,14.01,private_job
92,29.712006,adult,medium,2,26.55,government_job
78,44.414062,adult,high,1,30.34,private_job
80,30.734544,middle_aged,medium,2,41.23,unemployed
65,23.802196,young,medium,2,41.18,freelancer


In [33]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)